In [71]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ronikdedhia/next-word-prediction")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'next-word-prediction' dataset.
Path to dataset files: /kaggle/input/next-word-prediction


In [72]:
import os

In [73]:
txt_path = os.path.join(path,os.listdir(path)[0])

In [74]:
with open(txt_path,'r',encoding='charmap') as file:
    text=file.read()

In [75]:
text = text.lower()

In [76]:
import re

In [77]:
matches = re.findall(r'[a-z\s]',text)
text = ''.join(matches).strip()

In [78]:
text = ''.join(matches).strip()

In [79]:
import nltk
from nltk.stem import WordNetLemmatizer
lema = WordNetLemmatizer()

In [80]:
import nltk
nltk.download('wordnet')

lem_text = ' '.join([lema.lemmatize(word) for word in text.split()])

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [81]:
text_len = len(text)
lemtextlen = len(lem_text)
print(f'''length of simple text = {text_len}
       length of simple lem text = {lemtextlen}''')

length of simple text = 557883
       length of simple lem text = 547918


In [82]:
import tensorflow
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from nltk.tokenize import sent_tokenize


In [83]:
tok = Tokenizer()

In [84]:
import nltk
nltk.download('punkt_tab')
text = sent_tokenize(lem_text)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [85]:
tok.fit_on_texts(text)

In [86]:
seq= tok.texts_to_sequences(text)

In [87]:
len(seq)

1

In [88]:
tok_word_seq = tok.word_index

In [89]:
def data_generator(seq):
    for s in seq:
        for i in range(len(s) - 2):
            yield s[0 : i + 2]  # Ya s[i:i+2] jo bhi aapki requirement ho


# Istemal karne ka tarika:
# Isko direct loop mein chalayein, poori list ek sath RAM mein load nahi hogi
for batch in data_generator(seq):
    pass

In [90]:
text

['project gutenberg the adventure of sherlock holmes by arthur conan doyle this ebook is for the use of anyone anywhere at no cost and with almost no restriction whatsoever you may copy it give it away or reuse it under the term of the project gutenberg license included with this ebook or online at wwwgutenbergnet title the adventure of sherlock holmes author arthur conan doyle release date november ebook last updated may language english character set encoding utf start of this project gutenberg ebook the adventure of sherlock holmes produced by an anonymous project gutenberg volunteer and jose menendez cover the adventure of sherlock holmes by arthur conan doyle content i a scandal in bohemia ii the redheaded league iii a case of identity iv the boscombe valley mystery v the five orange pip vi the man with the twisted lip vii the adventure of the blue carbuncle viii the adventure of the speckled band ix the adventure of the engineer thumb x the adventure of the noble bachelor xi the 

In [91]:
seq

[[140,
  347,
  1,
  381,
  5,
  127,
  32,
  45,
  467,
  2043,
  2044,
  26,
  968,
  14,
  21,
  1,
  272,
  5,
  389,
  2045,
  19,
  40,
  1252,
  3,
  17,
  561,
  40,
  3047,
  3048,
  10,
  69,
  537,
  7,
  185,
  7,
  118,
  60,
  3049,
  7,
  257,
  1,
  450,
  5,
  1,
  140,
  347,
  668,
  2046,
  17,
  26,
  968,
  60,
  2047,
  19,
  4269,
  2434,
  1,
  381,
  5,
  127,
  32,
  4270,
  467,
  2043,
  2044,
  4271,
  753,
  4272,
  968,
  133,
  3050,
  69,
  3051,
  1040,
  538,
  246,
  4273,
  4274,
  694,
  5,
  26,
  140,
  347,
  968,
  1,
  381,
  5,
  127,
  32,
  1768,
  45,
  49,
  3052,
  140,
  347,
  1132,
  3,
  3053,
  3054,
  1544,
  1,
  381,
  5,
  127,
  32,
  45,
  467,
  2043,
  2044,
  1769,
  4,
  2,
  797,
  8,
  900,
  2435,
  1,
  562,
  669,
  2436,
  2,
  102,
  5,
  2048,
  3055,
  1,
  695,
  1371,
  451,
  3056,
  1,
  324,
  969,
  901,
  3057,
  1,
  56,
  17,
  1,
  1041,
  382,
  3058,
  1,
  381,
  5,
  1,
  468,
  1372,
  3059,
  1,
 

In [92]:
x = []
y = []

# Loop over the word sequence to create sliding-window n-grams
# For example, using a context sequence window of 5 words to predict the next word
single_seq = seq[0]
sequence_length = 10  # we will use a shorter window suitable for LSTM learning

for i in range(sequence_length, len(single_seq)):
    x.append(single_seq[i-sequence_length:i])
    y.append(single_seq[i])

In [93]:
x

[[140, 347, 1, 381, 5, 127, 32, 45, 467, 2043],
 [347, 1, 381, 5, 127, 32, 45, 467, 2043, 2044],
 [1, 381, 5, 127, 32, 45, 467, 2043, 2044, 26],
 [381, 5, 127, 32, 45, 467, 2043, 2044, 26, 968],
 [5, 127, 32, 45, 467, 2043, 2044, 26, 968, 14],
 [127, 32, 45, 467, 2043, 2044, 26, 968, 14, 21],
 [32, 45, 467, 2043, 2044, 26, 968, 14, 21, 1],
 [45, 467, 2043, 2044, 26, 968, 14, 21, 1, 272],
 [467, 2043, 2044, 26, 968, 14, 21, 1, 272, 5],
 [2043, 2044, 26, 968, 14, 21, 1, 272, 5, 389],
 [2044, 26, 968, 14, 21, 1, 272, 5, 389, 2045],
 [26, 968, 14, 21, 1, 272, 5, 389, 2045, 19],
 [968, 14, 21, 1, 272, 5, 389, 2045, 19, 40],
 [14, 21, 1, 272, 5, 389, 2045, 19, 40, 1252],
 [21, 1, 272, 5, 389, 2045, 19, 40, 1252, 3],
 [1, 272, 5, 389, 2045, 19, 40, 1252, 3, 17],
 [272, 5, 389, 2045, 19, 40, 1252, 3, 17, 561],
 [5, 389, 2045, 19, 40, 1252, 3, 17, 561, 40],
 [389, 2045, 19, 40, 1252, 3, 17, 561, 40, 3047],
 [2045, 19, 40, 1252, 3, 17, 561, 40, 3047, 3048],
 [19, 40, 1252, 3, 17, 561, 40, 3047, 

In [94]:
y

[2044,
 26,
 968,
 14,
 21,
 1,
 272,
 5,
 389,
 2045,
 19,
 40,
 1252,
 3,
 17,
 561,
 40,
 3047,
 3048,
 10,
 69,
 537,
 7,
 185,
 7,
 118,
 60,
 3049,
 7,
 257,
 1,
 450,
 5,
 1,
 140,
 347,
 668,
 2046,
 17,
 26,
 968,
 60,
 2047,
 19,
 4269,
 2434,
 1,
 381,
 5,
 127,
 32,
 4270,
 467,
 2043,
 2044,
 4271,
 753,
 4272,
 968,
 133,
 3050,
 69,
 3051,
 1040,
 538,
 246,
 4273,
 4274,
 694,
 5,
 26,
 140,
 347,
 968,
 1,
 381,
 5,
 127,
 32,
 1768,
 45,
 49,
 3052,
 140,
 347,
 1132,
 3,
 3053,
 3054,
 1544,
 1,
 381,
 5,
 127,
 32,
 45,
 467,
 2043,
 2044,
 1769,
 4,
 2,
 797,
 8,
 900,
 2435,
 1,
 562,
 669,
 2436,
 2,
 102,
 5,
 2048,
 3055,
 1,
 695,
 1371,
 451,
 3056,
 1,
 324,
 969,
 901,
 3057,
 1,
 56,
 17,
 1,
 1041,
 382,
 3058,
 1,
 381,
 5,
 1,
 468,
 1372,
 3059,
 1,
 381,
 5,
 1,
 1770,
 696,
 3060,
 1,
 381,
 5,
 1,
 1042,
 697,
 3061,
 1,
 381,
 5,
 1,
 798,
 1133,
 3062,
 1,
 381,
 5,
 1,
 1253,
 403,
 3063,
 1,
 381,
 5,
 1,
 754,
 849,
 4,
 2,
 797,
 8,
 900,
 4,


In [95]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
x = pad_sequences(x, maxlen=10, padding='post')

In [96]:
import numpy as np
y = np.array(y).reshape(-1, 1)

In [97]:
from tensorflow.keras import Sequential,models,layers,losses,metrics

In [98]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.23, random_state=42)

In [99]:
len(tok_word_seq)

7820

In [100]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Input(shape=(10, 1)),
    layers.LSTM(512, return_sequences=True),
    layers.LSTM(256),
    layers.Dense(8646, activation="softmax")
])

In [101]:
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [102]:
model.fit(x, y, validation_split=0.2, epochs=1)

2686/2686 ━━━━━━━━━━━━━━━━━━━━ 654s 242ms/step - accuracy: 0.0637 - loss: 6.3998 - val_accuracy: 0.0642 - val_loss: 6.4897


In [103]:
# Model structural summary dekhne ke liye:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_4 (LSTM)                   │ (None, 10, 512)        │     1,052,672 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 256)            │       787,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8646)           │     2,222,022 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,186,452 (46.49 MB)

 Trainable params: 4,062,150 (15.50 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 8,124,302 (30.99 MB)

In [104]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_5_real_words(seed_text):
    for _ in range(5):
        # Use 'tok' instead of 'word_tokenizer'
        token_list = tok.texts_to_sequences([seed_text])[0]
        # Pad to maxlen=10 to match your model's training shape
        token_list = pad_sequences([token_list], maxlen=10, padding='pre')
        # Reshape to (1, 10, 1) to match layers.Input(shape=(10, 1))
        token_list = np.reshape(token_list, (1, 10, 1))
        # Use 'model' instead of 'word_model'
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)[0]

        output_word = ""
        for word, index in tok.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

# Test with a proper word-level seed
test_seed = "sherlock holmes by arthur"
print("Input seed:", test_seed)
print("Predicted:", predict_5_real_words(test_seed))

Input seed: sherlock holmes by arthur
Predicted: sherlock holmes by arthur and the man of the


In [105]:
tok.word_index

{'the': 1,
 'a': 2,
 'and': 3,
 'i': 4,
 'of': 5,
 'to': 6,
 'it': 7,
 'in': 8,
 'that': 9,
 'you': 10,
 'he': 11,
 'wa': 12,
 'his': 13,
 'is': 14,
 'my': 15,
 'have': 16,
 'with': 17,
 'had': 18,
 'at': 19,
 'which': 20,
 'for': 21,
 'not': 22,
 'but': 23,
 'be': 24,
 'me': 25,
 'this': 26,
 'we': 27,
 'there': 28,
 'from': 29,
 'said': 30,
 'upon': 31,
 'holmes': 32,
 'so': 33,
 'him': 34,
 'her': 35,
 'she': 36,
 'all': 37,
 'your': 38,
 'very': 39,
 'no': 40,
 'been': 41,
 'on': 42,
 'what': 43,
 'one': 44,
 'by': 45,
 'then': 46,
 'are': 47,
 'were': 48,
 'an': 49,
 'would': 50,
 'when': 51,
 'out': 52,
 'mr': 53,
 'ha': 54,
 'up': 55,
 'man': 56,
 'do': 57,
 'could': 58,
 'if': 59,
 'or': 60,
 'into': 61,
 'who': 62,
 'little': 63,
 'will': 64,
 'some': 65,
 'see': 66,
 'now': 67,
 'down': 68,
 'may': 69,
 'our': 70,
 'should': 71,
 'they': 72,
 'u': 73,
 'well': 74,
 'am': 75,
 'hand': 76,
 'know': 77,
 'think': 78,
 'over': 79,
 'can': 80,
 'come': 81,
 'about': 82,
 'more': 8

In [107]:
# Use your custom prediction helper function with the text seed
predict_5_real_words("collection will remain freely")

'collection will remain freely and the man of the'